In [2]:
!pip -q install pypdf dotenv

In [3]:
import getpass
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"]=

#export LANGSMITH_API_KEY=""

'DUMMY_KEY_REDACTED'

In [4]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [5]:
pip install -qU langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

In [7]:
vector_1 = embeddings.embed_query(documents[0].page_content)

print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 1024

[-0.015638113, -0.016813638, -0.0003609887, -0.004475754, 0.008387622, -0.027549583, 0.03549157, 0.022143552, 0.04789924, 0.059988357]


In [8]:
pip install -qU langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec  # 👈 Added ServerlessSpec import
from langchain_pinecone import PineconeVectorStore

# 1. Initialize client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "pdf-textss"

# 2. Create the index
pc.create_index(
    name=index_name, 
    dimension=1024,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")  # This works now!
)


PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'f4ea21db0471712ffec5b004b832530f', 'date': 'Fri, 12 Jun 2026 06:14:02 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [ ]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

pc = Pinecone()
index = pc.Index(index_name)

vector_store = PineconeVectorStore(embedding=embeddings, index=index)

In [ ]:
# 3. Correctly initialize LangChain with the index name string
vector_store = PineconeVectorStore(
    index_name=index_name, 
    embedding=embeddings
)


In [ ]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "/home/mikealson/Documents/epfo/Xh3fivsfVVcdwHD_epm1-E1AFAiFzi_UVYV4_tM2Ja4.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))
#print(docs)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))
#print(all_splits)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)
ids

In [ ]:
results = vector_store.similarity_search(
    "mobile number"
)

print(results[0])

In [ ]:
results = await vector_store.asimilarity_search("What was the mobile number menctioned?")

print(results[0])
#llm->query->perfect answer

In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=1)
    serialized = "\n\n".join(
        (f"Source:\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [ ]:
# from langchain.agents import create_agent
# from langchain_ollama import ChatOllama

# model=ChatOllama(model="qwen3.5:0.8b")
# tools = [retrieve_context]
# # If desired, specify custom instructions
# prompt = (
#     "You are a helpful assistant. You have a tool called 'retrieve_context' "
#     "that searches PDF documents. "
#     "\n\nCRITICAL RULES:\n"
#     "1. If the user asks about document details, always call the 'retrieve_context' tool first.\n"
#     "2. Use the data from the tool to answer the question.\n"
#     "3. Keep your final answer short, simple, and direct. Use everyday words.\n"
#     "4. If you do not know the answer, just say 'I do not know'."
# )

# agent = create_agent(model, tools, system_prompt=prompt)

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

model=ChatOllama(model="qwen2.5-coder-64k:latest")
tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You are a helpful assistant. You have a tool called 'retrieve_context' "
    "that searches PDF documents. "
    "\n\nCRITICAL RULES:\n"
    "1. If the user asks about document details, always call the 'retrieve_context' tool first.\n"
    "2. Use the data from the tool to answer the question.\n"
    "3. Keep your final answer short, simple, and direct. Use everyday words.\n"
    "4. If you do not know the answer, just say 'I do not know'."
)

agent = create_agent(model, tools, system_prompt=prompt)

In [ ]:
# Invoke the agent using the correct 'messages' structure and your query
response = agent.invoke(
    {"messages": [{"role": "user", "content": "mobile number of the user"}]}
)

# Print the last message from the agent's output list
print(response["messages"][-1].content)


In [ ]:
agent.invoke({"promt":"mobile number of the user"})

In [ ]:
pip install -qU "langchain[google-genai]"

In [ ]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [ ]:
tools = [retrieve_context]
prompt = (
    "You are a helpful assistant. You have a tool called 'retrieve_context' "
    "that searches PDF documents. "
    "\n\nCRITICAL RULES:\n"
    "1. If the user asks about document details, always call the 'retrieve_context' tool first.\n"
    "2. Use the data from the tool to answer the question.\n"
    "3. Keep your final answer short, simple, and direct. Use everyday words.\n"
    "4. If you do not know the answer, just say 'I do not know'."
)

agent = create_agent(
    model=model, 
    tools=tools, 
    system_prompt=prompt
)


In [ ]:
model.invoke("hi")

In [ ]:
from langchain_core.utils.uuid import uuid7
config = {"configurable": {"thread_id": str(uuid7())}}
response = agent.invoke(
  {"messages": [("user", "what are the ten digit number you find  in the document ")]},

   # {"messages": [("user", "summarize the document/context you have")]},
    config=config
)
print(response)

In [ ]:
print(response)

In [ ]:
agent.invoke({"input":"mobile number of the user"})